In [ ]:
import pandas as pd 
from PIL import Image
import os



In [ ]:
DATA_PATH = os.environ.get("DATA_PATH","./data/dataset")

df = pd.read_csv(os.path.join(DATA_PATH,'manifest.csv'))
df = df[['relative_path','filename']]

In [32]:
dftrain = df[df['relative_path'].str.contains('train')]
dftest = df[df['relative_path'].str.contains('test')]

In [42]:
import numpy as np

In [ ]:
def make_bounding_boxes(df,outputpath):
    os.makedirs(outputpath, exist_ok=True)
    for i in range(len(df)):
        path,filename = df.iloc[i]
        name,_ = os.path.splitext(filename)
        img = Image.open(os.path.join('./data/dataset',path))
        img = np.array(img)
        img[img<=127] = 0
        img[img>127] = 1
        imgh = img.shape[0]
        imgw = img.shape[1]
        mask = np.argwhere(img==1)
        if mask.shape[0]==0:
            continue #No Tumor Mask
        min_y,max_y = np.min(mask[:,0]),np.max(mask[:,0])
        min_x,max_x = np.min(mask[:,1]),np.max(mask[:,1])
        text_file = f"{0} {((max_x+min_x)/2)/imgw} {((max_y+min_y)/2)/imgh} {(max_x-min_x)/imgw} {(max_y-min_y)/imgh}\n"
        with open(os.path.join(outputpath,f"{name}.txt"),'w') as f:
            f.write(text_file)
    

In [ ]:

OUTPUT_PATH = os.environ.get("OUTPUT_PATH","./masks")

make_bounding_boxes(dftrain,OUTPUT_PATH)
make_bounding_boxes(dftest,OUTPUT_PATH)